# Counting statistics and CCDs

We have seen in the videos how CCDs operate in the optical, with roughly one photoelectron produced per photon. The amount of charge collected in a single pixel (or a region of size the point-spread function) is then proportional to the intensity. The typical energy required to liberate one photoelectron is 1.2 eV.

We can also use CCDs in the X-rays, e.g. on the [_Chandra X-ray Observatory_](https://chandra.harvard.edu/about/) ACIS detectors. These detectors operate in the X-ray band, 0.5-10 keV (See also [2018 assignment Q5](https://learning.monash.edu/course/view.php?id=40326&section=5])  which covers this instrument).

Imagine a 1 keV photon intercepting one of the ACIS CCDs

How many photoelectrons would the photon liberate from the Si matrix?

In [ ]:
# Answer here

Assuming all (most of?) the energy of the photon is translated to the CCD, we can then estimate the _photon energy_ from the number of photoelectrons.

With what precision? (i.e. what is the uncertainty?)

In [ ]:
# Answer here

What happens in a pixel when two photons fall during the same exposure?

How would we interpret the energy?

In [ ]:
# Answer here

To avoid this "pileup" issue, we need to ensure that at most one photon per pixel is received by the detector, and set short "frame" (exposure) times to ensure this, at most 1.7 s. 

We also need to limit our observation targets to faint sources.

Imagine an observation lasting 26 ks ($26\times10^3$ s), in which 3400 photons are detected. What is the average count rate?

In [ ]:
# Answer here

Assuming all the flux falls on a single pixel, what is the probability that more than one photon falls on the detector within one 1.7 s frame time?

_Hint:_ you may need to make use of the Poisson distribution, which gives the probability of detecting $n$ counts in a given sample of a process with mean rate $\lambda$:

$P_n = \frac{\lambda^n\exp(-\lambda)}{n!}$

In [ ]:
# Answer here

# GRB 230911A counterpart discovery

This gamma-ray burst was detected on 2023 September 11 03:09:32 UT with the _Fermi_ satellite, and a ~19 mag possible optical counterpart was detected with the GOTO telescope in Siding Spring, NSW ([Belkin et al., 2024](https://ui.adsabs.harvard.edu/abs/2024RNAAS...8....6B).

The position from _Fermi_ is RA = 59.8, Dec = -34.4 (J2000 degrees, equivalent to J2000 03h 59m, -34d 23'), with a statistical uncertainty of 4.1 degrees ([GCN 34652](https://gcn.nasa.gov/circulars/34652).

Why is the spatial uncertainty so poor? (Think about what you know about telescopes and how a telescope for gamma rays might be constructed!)

Imagine how many transients at 19th mag or greater you might find in a circle of radius 4.1 degrees... how _confident_ could we bet that the optical counterpart is related to the gamma-ray burst?

For a general object this question of _association_ can be handled statistically based on position alone, and/or from the properties of the object, perhaps including intensity.

For example if we know the average concentration/rate of such objects per unit sky area, we can estimate the chance of an _unrelated_ object popping up around the same time (the _null hypothesis_). 

The assignment question hypothesises an ultraluminous X-ray source detected by _Chandra_, with the position known to 1.3". Within the same region, and IR counterpart is found; but the field is crowded with IR sources, with a spatial density of 100/arcmin^2

What is the probability that the IR source is associated with the X-ray?

In [ ]:
# Answer here

For the GRB optical counterpart, unfortunately, we don't know when the candidate source appeared (our last observation of the field was back in 2020!)

And, estimating the average rate of 19-mag transients in GOTO fields is a bit challenging...

But we can look at the properties of the source, as we know what a GRB lightcurve looks like. And this object is declining rapidly, as we'd expect from the optical afterglow.

![image.png](https://content.cld.iop.org/journals/2515-5172/8/1/6/revision1/rnaasad1876f1_lr.jpg)

# GRB 230911A light curve fitting

Here we provide the original data for the GRB optical afterglow, for you to try to perform your own fit and replicate the analysis in the paper

In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import t

# --- Data: t (hours), mag, mag_err ---
data = np.array([
    [1.17, 19.222, 0.100],
    [1.77, 19.183, 0.088],
    [2.32, 19.714, 0.155],
    [2.70, 19.438, 0.216],
    [11.82, 20.615, 0.170],
    [12.97, 20.767, 0.187],
    [37.98, 21.533, 0.303],
])

# --- Extract columns ---
t_hours = data[:, 0]
mag = data[:, 1]
mag_err = data[:, 2]

# --- Convert to days ---
t_days = t_hours / 24.0

# --- Constants ---
zp_flux = 3624.05  # Zeropoint in your system

# --- Convert to flux ---
flux = 10 ** (np.log10(zp_flux) - 0.4 * mag)

# Asymmetric uncertainties
flux_plus = 10 ** (np.log10(zp_flux) - 0.4 * (mag - mag_err))
flux_minus = 10 ** (np.log10(zp_flux) - 0.4 * (mag + mag_err))
flux_err = (flux_plus - flux) + (flux - flux_minus)
flux_err /= 2

# -----------------------------------------------------------------
# Define and fit your curve here, and evaluate it at times t_fit

# --- Create fit curve ---
t_fit = np.logspace(np.log10(min(t_days)*0.8), np.log10(max(t_days)*1.2), 200)
flux_fit = None
# flux_fit = [insert expression to evaluate fit model here]
# -----------------------------------------------------------------

# --- Plot: Magnitude space ---
plt.figure(figsize=(8, 5))
plt.errorbar(t_days, mag, yerr=mag_err, fmt='o', label='L band', capsize=3)
plt.gca().invert_yaxis()
plt.xscale('log')
plt.xlabel('Time since trigger [days]')
plt.ylabel('Magnitude')
plt.grid(True)
plt.legend()
plt.tight_layout()

if flux_fit is not None:
    mag_fit = (np.log10(zp_flux) - np.log10(flux_fit)) / 0.4
    plt.plot(t_fit, mag_fit, '--', label=f'SPL fit: $\\alpha$ = {alpha_fit:.2f} ± {alpha_err:.2f}')

plt.show()

# --- Plot: Flux space ---
plt.figure(figsize=(8, 5))
plt.errorbar(t_days, flux, yerr=flux_err, fmt='o', capsize=3, label='L band')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time since trigger [days]')
plt.ylabel('Flux density [Jy]')
#plt.grid(True, which='both')
plt.grid(True)
plt.legend()
plt.tight_layout()
if flux_fit is not None:
    plt.plot(t_fit, flux_fit, '--', label=f'SPL fit: $\\alpha$ = {alpha_fit:.2f} ± {alpha_err:.2f}')

plt.show()


# Fit with confidence band

Here we illustrate the effect of errors in the fit parameters, and generate a 95% confidence band to overplot. 

Try varying the `alpha` value to generate e.g. a 1-sigma confidence band instead

In [ ]:
# results of original fit

popt = np.array([1.28803344e-05, 6.14957377e-01])
pcov = np.array([[ 2.46755456e-12, -7.19845335e-08],
                 [-7.19845335e-08,  2.49850941e-03]])
A_fit, alpha_fit = popt
A_err, alpha_err = np.sqrt(np.diag(pcov))
flux_fit = A_fit * t_fit**(-alpha_fit)

In [ ]:
# --- Calculate 95% confidence interval for the fit curve ---
alpha = 0.05  # 95% confidence
dof = len(t_days) - len(popt)  # degrees of freedom
tval = t.ppf(1 - alpha / 2., dof)

# Jacobian matrix: partial derivatives of model wrt params
def jacobian(t, A, alpha):
    dA = t ** (-alpha)
    dalpha = -A * t ** (-alpha) * np.log(t)
    return np.vstack((dA, dalpha)).T

# Get standard deviation of the fit
J = jacobian(t_fit, *popt)
fit_var = np.einsum("ij,jk,ik->i", J, pcov, J)  # variance at each t
fit_std = np.sqrt(fit_var)
flux_fit_upper = flux_fit + tval * fit_std
flux_fit_lower = flux_fit - tval * fit_std

# --- Plot with confidence region in flux space ---
plt.figure(figsize=(8, 5))
plt.errorbar(t_days, flux, yerr=flux_err, fmt='o', capsize=3, label='L band')
plt.plot(t_fit, flux_fit, '--', label=f'Fit: A={A_fit:.3g}±{A_err:.3g}, α={alpha_fit:.2f}±{alpha_err:.2f}')
plt.fill_between(t_fit, flux_fit_lower, flux_fit_upper, color='orange', alpha=0.3, label='95% Confidence Band')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time [days]')
plt.ylabel('Flux density [Jy]')
plt.legend()
#plt.grid(True, which='both')
plt.grid(True)
plt.tight_layout()
plt.show()